In [1]:
import tensorflow as tf
from tensorflow.keras import layers, Model, Input, initializers

import numpy as np

In [3]:
class SelfAttentionBlock(layers.Layer):
    def __init__(self, emb_dim, name_prefix=''):
        super().__init__(name=name_prefix)
        
        self.attn = layers.MultiHeadAttention(
            num_heads=2,
            key_dim=emb_dim,
            dropout=0.1,
            name=f'{name_prefix}_attn'
        )

    # def call(self, x, mask=None):
    def call(self, x):
        attn_output = self.attn(x, x)
        return attn_output
    
class RankingModel(Model):
    def __init__(self, embedding_dim):
        super().__init__()
        
        # self.inputs = layers.Input(shape=(None,), ragged=False, dtype=tf.int32)

        self.attn_self = SelfAttentionBlock(embedding_dim, name_prefix='self_attenction_block_1')
        
        self.dense1 = layers.Dense(16, activation='relu', name = "dense1")
        self.score_layer = layers.Dense(1)  # output predicted score per item

    def call(self, inputs):
        doc_features = inputs['doc_features']  # shape: [batch_size, list_size, num_doc_features]

        x = self.attn_self(doc_features ) # x = self.attn_self(masked , mask_tensor)
        x = self.dense1(x)  
        scores = self.score_layer(x)  # [batch_size, list_size, 1]
        scores = tf.squeeze(scores, axis=-1)  # [batch_size, list_size]

        return scores

In [33]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

def selfAttentionBlock(inputs, emb_dim, name_prefix=''):
    """Functional block implementing MultiHeadAttention."""
    attn_output = layers.MultiHeadAttention(
        num_heads=2,
        key_dim=emb_dim,
        dropout=0.1,
        name=f'{name_prefix}_attn'
    )(inputs, inputs)  # Self-attention: query=inputs, key=inputs
    return attn_output

def build_ranking_model(embedding_dim):
    # Define input layer (ragged=False here, but can be ragged=True if needed)
    doc_features_in = Input( shape=(None, embedding_dim), name="doc_features", dtype=tf.float32, ragged=True )

    # Self-attention block
    x = selfAttentionBlock(doc_features_in, embedding_dim, name_prefix='self_attention_block_1')

    # Dense layers
    x_dense = layers.Lambda(lambda y: y.to_tensor(), output_shape=(None, embedding_dim))(x) 
    x = layers.Dense(16, activation='relu', name="dense1")(x_dense)
    
    scores = layers.Dense(1, name="score_layer")(x)  # [batch, list_size, 1]
    scores = layers.Lambda(lambda t: tf.squeeze(t, axis=-1), name="squeeze_scores")(scores)  # [batch, list_size]

    # Build the model
    model = Model(
        # inputs={"doc_features": doc_features_in},
        inputs=doc_features_in,
        outputs=scores,
        name="RankingModel"
    )
    return model

# Example usage
embedding_dim = 4
model_fun = build_ranking_model(embedding_dim)
model_fun.summary()

Model: "RankingModel"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ doc_features        │ (None, None, 4)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ self_attention_blo… │ (None, None, 4)   │        156 │ doc_features[0][… │
│ (MultiHeadAttentio… │                   │            │ doc_features[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lambda_3 (Lambda)   │ (None, None, 4)   │          0 │ self_attention_b… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense1 (Dense)      │ (None, None, 16)  │         80 │ lambda_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ score_layer (Dense) │ (None, None, 1)   │         17 │ dense1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ squeeze_scores      │ (None, None)      │          0 │ score_layer[0][0] │
│ (Lambda)            │                   │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 253 (1012.00 B)

 Trainable params: 253 (1012.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
# Example data
uqids = tf.ragged.constant([1, 2, 3], dtype=tf.float32)
docs_ragged = tf.ragged.constant([[[0, 1, 2, 3],
  [4, 5, 6, 7]], [[8, 9, 10, 11],
                  [12, 13, 14, 15],
                  [16, 17, 18, 19]], [[20, 21, 22, 23]]],ragged_rank=1, dtype=tf.float32)

labels_ragged = tf.ragged.constant([[1.0, 0.0], [1.0, 1.0, 0.0], [1.0]], ragged_rank=1, dtype=tf.float32)

# docs_ragged, labels_ragged, uqids = group_by_query_id_ragged(X, y, qids, debug=True)

print("\nDocs RaggedTensor:", docs_ragged)
print("\nDocs RaggedTensor.shape:", docs_ragged.shape)
print("Labels RaggedTensor:", labels_ragged)
print("Labels shape:", labels_ragged.shape)
print("uqids :", uqids)

test_x = {
            # "user_id": uqids,
            'doc_features': docs_ragged
        }



Docs RaggedTensor: <tf.RaggedTensor [[[0.0, 1.0, 2.0, 3.0],
  [4.0, 5.0, 6.0, 7.0]], [[8.0, 9.0, 10.0, 11.0],
                          [12.0, 13.0, 14.0, 15.0],
                          [16.0, 17.0, 18.0, 19.0]],
 [[20.0, 21.0, 22.0, 23.0]]]>

Docs RaggedTensor.shape: (3, None, 4)
Labels RaggedTensor: <tf.RaggedTensor [[1.0, 0.0], [1.0, 1.0, 0.0], [1.0]]>
Labels shape: (3, None)
uqids : tf.Tensor([1. 2. 3.], shape=(3,), dtype=float32)


In [38]:
uqids = np.array([1, 2, 3], dtype=np.float32)
docs_ragged = np.array([[[0, 1, 2, 3],
  [4, 5, 6, 7]], [[8, 9, 10, 11],
                  [12, 13, 14, 15],
                  [16, 17, 18, 19]], [[20, 21, 22, 23]]],dtype=np.float32)

labels_ragged = np.array([[1.0, 0.0], [1.0, 1.0, 0.0], [1.0]], dtype=np.float32)

# docs_ragged, labels_ragged, uqids = group_by_query_id_ragged(X, y, qids, debug=True)

print("\nDocs RaggedTensor:", docs_ragged)
print("\nDocs RaggedTensor.shape:", docs_ragged.shape)
print("Labels RaggedTensor:", labels_ragged)
print("Labels shape:", labels_ragged.shape)
print("uqids :", uqids)

ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (3,) + inhomogeneous part.

In [34]:
model_fun.compile(
            optimizer=tf.keras.optimizers.Adagrad(
                learning_rate=0.1
            ),
            loss=custom_loss
        )
print(docs_ragged.shape)
print(labels_ragged.shape)
model_fun.fit(docs_ragged, labels_ragged.to_tensor(), epochs=50, batch_size=5)

(3, None, 4)
(3, None)
Epoch 1/50


TypeError: Exception encountered when calling EinsumDense.call().

[1mTensors in list passed to 'inputs' of 'Einsum' Op have types [<NOT CONVERTIBLE TO TENSOR>, float32] that don't all match.[0m

Arguments received by EinsumDense.call():
  • inputs=tf.Tensor(shape=(None, None, 4), dtype=float32)
  • training=True

In [14]:
tf.__version__


'2.19.0'

In [35]:
import tensorflow as tf
print(tf.keras.__version__)

3.10.0


In [36]:
import keras
print(keras.__version__)

3.10.0


In [9]:

def custom_loss(y_true, y_pred, debug = False):
    return tf.reduce_mean(y_true - y_pred)


model = RankingModel( 32)
model.compile(
            optimizer=tf.keras.optimizers.Adagrad(
                learning_rate=0.1
            ),
            loss=custom_loss
        )

model.fit(test_x, labels_ragged, epochs=50, batch_size=5)


Epoch 1/50


/opt/anaconda3/envs/graph_temporal/lib/python3.12/site-packages/keras/src/layers/layer.py:1474: UserWarning: Layer 'ranking_model_1' looks like it has unbuilt state, but Keras is not able to trace the layer `call()` in order to build it automatically. Possible causes:
1. The `call()` method of your layer may be crashing. Try to `__call__()` the layer eagerly on some test input first to see if it works. E.g. `x = np.random.random((3, 4)); y = layer(x)`
2. If the `call()` method is correct, then you may need to implement the `def build(self, input_shape)` method on your layer. It should create all variables used by the layer (e.g. by calling `layer.build()` on all its children layers).
Exception encountered: ''Exception encountered when calling EinsumDense.call().

Tensors in list passed to 'inputs' of 'Einsum' Op have types [<NOT CONVERTIBLE TO TENSOR>, float32] that don't all match.

Arguments received by EinsumDense.call():
  • inputs=tf.Tensor(shape=(None, None, 4), dtype=float32)
  

TypeError: Exception encountered when calling EinsumDense.call().

[1mTensors in list passed to 'inputs' of 'Einsum' Op have types [<NOT CONVERTIBLE TO TENSOR>, float32] that don't all match.[0m

Arguments received by EinsumDense.call():
  • inputs=tf.Tensor(shape=(None, None, 4), dtype=float32)
  • training=None